<!-- WARNING: THIS FILE WAS AUTOGENERATED! DO NOT EDIT! -->

The exported graph is the inference network. Dropout is gone. Each Conv-BN-Swish stays unfused, and each LayerNorm is a learned scale and bias. Two global transformer blocks sit between a 1×7 depthwise conv and an additive 1×1 skip. The NRTR decoder is not in the checkpoint.

In [4]:
#| echo: false
#| output: asis
show_doc(LayerNorm)

---

[source](https://github.com/alix559/solid_lens/blob/main/solid_lens/layers/svtr.py#L24){target="_blank" style="float:right; font-size:smaller"}

### LayerNorm

```python
def LayerNorm(
    channels:int, eps:float
)->None:
```

*Scale and bias. The checkpoint stores them as a Mul followed by an Add.*

The checkpoint does not contain a LayerNorm op. It contains the scale and the bias, and `weights` finds those as a Mul followed by an Add. This module is the pair those tensors belong to.

In [6]:
#| echo: false
#| output: asis
show_doc(ConvBNSwish)

---

[source](https://github.com/alix559/solid_lens/blob/main/solid_lens/layers/svtr.py#L38){target="_blank" style="float:right; font-size:smaller"}

### ConvBNSwish

```python
def ConvBNSwish(
    in_channels:int, out_channels:int, eps:float
)->None:
```

*1x1 projection used by the skip and the channel reduction.*

In [8]:
#| echo: false
#| output: asis
show_doc(LocalMixer)

---

[source](https://github.com/alix559/solid_lens/blob/main/solid_lens/layers/svtr.py#L51){target="_blank" style="float:right; font-size:smaller"}

### LocalMixer

```python
def LocalMixer(
    channels:int, kernel:int, eps:float
)->None:
```

*1xk depthwise conv along the text width, added back onto its input.*

The local mixer is a 1×7 depthwise conv along the sequence, then batch-norm and SiLU, added back to its input. It does not mix height. By the time it runs, the map is one row tall.

## Attention

In [10]:
#| echo: false
#| output: asis
show_doc(SVTRBlock)

---

[source](https://github.com/alix559/solid_lens/blob/main/solid_lens/layers/svtr.py#L72){target="_blank" style="float:right; font-size:smaller"}

### SVTRBlock

```python
def SVTRBlock(
    dim:int, num_heads:int, mlp_ratio:float, eps:float
)->None:
```

*Pre-norm global attention and a Swish MLP. Dropout is identity here.*

The scale multiplies the query only. That is how the exported graph is written, and it is not the same op as scaling the scores after the matmul when the dtype is not exact. The MLP is SiLU, not GELU. Dropout is absent, so it is not in the module.

In [12]:
#| echo: false
#| output: asis
show_doc(LightSVTR)

---

[source](https://github.com/alix559/solid_lens/blob/main/solid_lens/layers/svtr.py#L119){target="_blank" style="float:right; font-size:smaller"}

### LightSVTR

```python
def LightSVTR(
    in_channels:int, dim:int, depth:int, num_heads:int, mlp_ratio:float, local_kernel:int, eps:float,
    final_eps:float
)->None:
```

*Local conv, global blocks, then an additive skip of the backbone features.*

The skip is constructed first because the exporter emits it first. `final_eps` is 1e-6 and applies only to the norm after the last block. The blocks themselves use `eps`, which is 1e-5.